In [3]:
import pandas as pd
import sqlite3

In [4]:
accepted_file = 'accepted_2007_to_2018Q4.csv'
rejected_file = 'rejected_2007_to_2018Q4.csv'

In [ ]:


df_accepted = pd.read_csv(accepted_file)
print("Accepted Loans Dataset:")
print(df_accepted.info())
print(df_accepted.head())




KeyboardInterrupt: 

In [5]:
df_accepted['loan_status_label'] = 'accepted'

# Convert date columns to datetime (example for a few columns)
date_cols_accepted = ['issue_d', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']
for col in date_cols_accepted:
    if col in df_accepted.columns:
        df_accepted[col] = pd.to_datetime(df_accepted[col], errors='coerce')

# Clean term column: extract numeric value from string (e.g., "36 months" -> 36)
df_accepted['term_numeric'] = df_accepted['term'].str.extract('(\d+)').astype(float)

# Optionally, create a quarter indicator for temporal analysis
if 'issue_d' in df_accepted.columns:
    df_accepted['issue_quarter'] = df_accepted['issue_d'].dt.to_period('Q')
    # Convert Period objects to strings
    df_accepted['issue_quarter'] = df_accepted['issue_quarter'].astype(str)

/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_9046/2693059868.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_accepted[col] = pd.to_datetime(df_accepted[col], errors='coerce')
/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_9046/2693059868.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_accepted[col] = pd.to_datetime(df_accepted[col], errors='coerce')
/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_9046/2693059868.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_accepted[col] = pd.to_datetime(df_accepted[col], errors='c

In [5]:

df_rejected = pd.read_csv(rejected_file)
print("\nRejected Loans Dataset:")
print(df_rejected.info())
print(df_rejected.head())


Rejected Loans Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27648741 entries, 0 to 27648740
Data columns (total 9 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Amount Requested      float64
 1   Application Date      object 
 2   Loan Title            object 
 3   Risk_Score            float64
 4   Debt-To-Income Ratio  object 
 5   Zip Code              object 
 6   State                 object 
 7   Employment Length     object 
 8   Policy Code           float64
dtypes: float64(3), object(6)
memory usage: 1.9+ GB
None
   Amount Requested Application Date                        Loan Title  \
0            1000.0       2007-05-26  Wedding Covered but No Honeymoon   
1            1000.0       2007-05-26                Consolidating Debt   
2           11000.0       2007-05-27       Want to consolidate my debt   
3            6000.0       2007-05-27                           waksman   
4            1500.0       2007-05-27             

In [6]:
df_rejected['Application Date'] = pd.to_datetime(df_rejected['Application Date'], errors='coerce')

# Clean 'Debt-To-Income Ratio': remove '%' and convert to float
df_rejected['Debt-To-Income Ratio'] = df_rejected['Debt-To-Income Ratio'].str.replace('%', '', regex=False).astype(float)

# Rename columns for consistency (optional)
df_rejected = df_rejected.rename(columns={
    'Amount Requested': 'loan_amnt',
    'Application Date': 'application_date',
    'Loan Title': 'loan_title',
    'Risk_Score': 'risk_score',
    'Debt-To-Income Ratio': 'dti',
    'Zip Code': 'zip_code',
    'State': 'state',
    'Employment Length': 'emp_length',
    'Policy Code': 'policy_code'
})

if 'application_date' in df_rejected.columns:
    df_rejected['application_date'] = pd.to_datetime(df_rejected['application_date'], errors='coerce')
    df_rejected['application_quarter'] = df_rejected['application_date'].dt.to_period('Q').astype(str)


In [6]:
# Define the database filename
db_filename = 'loans.db'

# Establish a connection to the SQLite database (it will be created if it doesn't exist)
conn = sqlite3.connect(db_filename)

# Write the accepted loans DataFrame to a table named 'accepted_loans'
df_accepted.to_sql('accepted_loans', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print("Database created and tables loaded successfully!")

Database created and tables loaded successfully!


In [7]:
# Define the database filename
db_filename = 'loans.db'

# Establish a connection to the SQLite database (it will be created if it doesn't exist)
conn = sqlite3.connect(db_filename)

# Write the accepted loans DataFrame to a table named 'accepted_loans'
df_rejected.to_sql('rejected_loans', conn, if_exists='replace', index=False, chunksize=10000)

conn.commit()
conn.close()

print("Database created and tables loaded successfully!")

Database created and tables loaded successfully!
